# 技能1 · Day 4 上机：多模态融合与跨域对齐

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 实现多模态融合三策略（早融合/中融合/晚融合），理解各策略的适用场景与优劣
2. 从零实现对比学习损失（InfoNCE + CLIP对称损失），理解温度参数对分布的影响
3. 用 **transformers CLIP** 实现图文检索（产品图-文案相似度 + top-k检索）
4. 用 **transformers BLIP** 实现图文理解（自动描述生成 + VQA视觉问答）
5. 用 CLIP 实现零样本分类，理解跨域对齐的原理与营销应用
6. 设计企业级多模态架构（广告创意图文匹配系统），评估各模块延迟与瓶颈


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ CLIP/BLIP 模型首次运行需从 HuggingFace 下载：
> - `openai/clip-vit-base-patch32`（~600MB）
> - `Salesforce/blip-image-captioning-base`（~250MB）
> - `Salesforce/blip-vqa-base`（~250MB）
> 如网络受限，可设置 `HF_ENDPOINT=https://hf-mirror.com` 使用镜像。

In [ ]:
# !pip install transformers torch pillow numpy -q
# CLIP/BLIP模型首次运行需下载：
#   openai/clip-vit-base-patch32 (~600MB)
#   Salesforce/blip-image-captioning-base (~250MB)
#   Salesforce/blip-vqa-base (~250MB)

## 1. 数据集背景与营销映射

**营销场景**：多模态营销内容融合与对齐 -- 将产品图片、文案描述、结构化属性融合为统一表示。

| 模态 | 营销数据 | 编码模型 | 维度 |
|------|---------|---------|------|
| 文本 | 产品描述/广告文案 | sentence-transformers / CLIP文本编码器 | 384-512 |
| 图像 | 产品图片/广告创意 | CLIP-ViT / ResNet | 512-2048 |
| 结构化 | 价格/评分/库存 | MLP | 16-128 |

**核心任务**：
1. **融合策略**：如何将异构模态融合为统一表示？
2. **对齐**：如何让图文在共享空间中对齐（CLIP对比学习）？
3. **检索**：给定文案，如何检索最匹配的产品图片？
4. **理解**：如何让模型"看懂"产品图片（BLIP自动描述+VQA）？

> 💡 本上机使用 PIL 生成模拟产品图片，代码无需下载外部图片。实际项目中替换为真实产品图片即可。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

# 工具函数：生成模拟产品图片（无需下载外部图片）
def make_product_image(color, label="PRODUCT"):
    img = Image.new('RGB', (224, 224), color)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 24)
    except:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), label, font=font)
    x = (224 - (bbox[2] - bbox[0])) // 2
    y = (224 - (bbox[3] - bbox[1])) // 2
    draw.text((x, y), label, fill='white', font=font)
    return img

print("环境就绪。torch:", torch.__version__)

## 2. 多模态融合三策略

企业拥有多种数据（文本/图像/结构化）时，如何融合为统一表示？

| 策略 | 融合层 | 优势 | 劣势 | 营销场景 |
|------|--------|------|------|---------|
| 早融合 | 特征层 | 学习模态间交叉特征 | 要求模态同时可用 | 产品推荐（图文+价格+评分） |
| 中融合 | 注意力层 | 动态权重，自适应 | 计算量较大 | 跨模态注意力（文案关注图片区域） |
| 晚融合 | 决策层 | 模态解耦，可独立训练 | 无法捕捉交叉特征 | 多渠道归因（搜索/展示/社交） |

**关键公式**：
- 早融合：`z = MLP([z_text; z_image; z_struct])`
- 中融合（注意力）：`alpha_i = softmax(W * y_i), y = sum(alpha_i * y_i)`
- 晚融合：`y = w1*y_text + w2*y_image + w3*y_struct`

> 详见独立教材 3.4.1 节（多模态表示融合三种策略）

In [ ]:
# TODO 1：多模态融合三策略实现（早融合/中融合/晚融合）
# 提示：用 torch.nn 实现三种融合策略，输入为文本/图像/结构化属性的embedding
#   早融合：拼接所有特征 -> MLP
#   中融合：跨模态注意力（text query, image key/value）
#   晚融合：各模态独立预测 -> 学习权重加权
# 要求：实现3个类，用合成数据测试，打印输出shape

# ===== 你的代码 =====

# 早融合
class EarlyFusion(nn.Module):
    # TODO: 实现 __init__ 和 forward
    pass

# 中融合（跨模态注意力）
class CrossModalAttentionFusion(nn.Module):
    # TODO: 实现 __init__ 和 forward
    pass

# 晚融合
class LateFusion(nn.Module):
    # TODO: 实现 __init__ 和 forward
    pass

# 测试（取消注释后实现）
# text_emb = torch.randn(4, 128)
# image_emb = torch.randn(4, 256)
# struct_emb = torch.randn(4, 16)
# TODO: 实例化3个模型，前向传播，打印输出shape

## 3. 对比学习（Contrastive Learning）

对比学习是 CLIP/SimCLR 等模型的核心技术：**通过拉近正样本对、推远负样本对来学习表示**。

**InfoNCE 损失**：
```
L = -log[ exp(sim(z, z+)/tau) / (exp(sim(z, z+)/tau) + sum(exp(sim(z, z-)/tau))) ]
```

- `z`：锚点（anchor），`z+`：正样本，`z-`：负样本
- `tau`：温度参数（tau小->分布尖锐，tau大->分布平坦）

**CLIP 的对称损失**：`L = (L_img2text + L_text2img) / 2`，双向对比。

**为什么对比学习有效**：不需要标签就能学到好的表示 -- 只要能定义"相似"和"不相似"。

**营销应用**：产品图片-文案对齐 -- 匹配的图文对为正样本，不匹配的为负样本。

> 详见独立教材 3.4.2 节（对比学习原理）

In [ ]:
# TODO 2：对比学习实现（InfoNCE loss + CLIP对称损失）
# 提示：实现两个损失函数
#   info_nce_loss(anchor, positive, negatives, temperature): 标准InfoNCE
#   clip_loss(image_features, text_features, temperature): CLIP对称损失
# 数学：L = -log[exp(sim(z,z+)/tau) / (exp(sim(z,z+)/tau) + sum(exp(sim(z,z-)/tau)))]
# 要求：用合成数据测试，观察温度参数对损失的影响

# ===== 你的代码 =====

def info_nce_loss(anchor, positive, negatives, temperature=0.07):
    # TODO: 实现InfoNCE损失
    # anchor: (batch, dim), positive: (batch, dim), negatives: (batch, num_neg, dim)
    pass

def clip_loss(image_features, text_features, temperature=0.07):
    # TODO: 实现CLIP对称对比损失
    # image_features: (batch, dim), text_features: (batch, dim)
    pass

# 测试（取消注释后实现）
# anchor = torch.randn(4, 128)
# positive = anchor + 0.1 * torch.randn(4, 128)
# negatives = torch.randn(4, 10, 128)
# TODO: 计算loss，测试不同temperature

## 4. CLIP：图文对齐的里程碑

CLIP（Contrastive Language-Image Pre-training，OpenAI 2021）用对比学习将图像和文本对齐到同一向量空间。

**核心架构**：双塔 -- 图像编码器（ViT）+ 文本编码器（Transformer），各自编码后投影到共享空间。

**关键 API**（transformers 库）：
- `CLIPModel.get_image_features()` -> 图像 embedding
- `CLIPModel.get_text_features()` -> 文本 embedding
- `CLIPModel(**inputs)` -> `outputs.logits_per_image`（相似度矩阵）

**从 CLIP 到 GPT-4o 的演进**：
| 阶段 | 模型 | 核心 |
|------|------|------|
| 对比学习对齐 | CLIP (2021) | 双塔+对比损失 |
| 视觉-语言预训练 | BLIP-2 (2023) | Q-Former桥接ViT+LLM |
| 原生多模态 | GPT-4o (2024) | 端到端统一token空间 |
| 开源多模态 | LLaVA (2024) | CLIP-ViT + 投影层 + LLM |

> 详见独立教材 3.2.3 节（从CLIP到GPT-4o的多模态演进）

In [ ]:
# TODO 3：CLIP图文检索（transformers CLIPModel）
# 提示：用 CLIPModel + CLIPProcessor 计算图文相似度，实现top-k检索
#   1. 生成产品图片（用make_product_image）
#   2. 用CLIP编码图片和文本
#   3. 计算余弦相似度矩阵
#   4. 给定文本query，检索最匹配的图片
# 营销场景：用户搜索"红色口红" -> 返回最匹配的产品图片
# 模型：openai/clip-vit-base-patch32

# ===== 你的代码 =====
from transformers import CLIPProcessor, CLIPModel

# TODO: 加载模型
# model = ...
# processor = ...

# TODO: 生成产品图片和文案
# product_images = [...]
# product_texts = [...]

# TODO: 编码图片和文本，计算相似度
# TODO: 实现top-k检索

## 5. BLIP-2：图文理解的突破

BLIP-2（Salesforce 2023）用 Q-Former 桥接冻结的视觉编码器和冻结的 LLM，实现图文理解。

**架构**：冻结ViT -> Q-Former（学习查询token）-> 冻结LLM

**与CLIP的区别**：
- CLIP 只做对齐（相似度），不生成文本
- BLIP-2 能生成描述、回答问题（生成式）

**API**（transformers库）：
- BLIP-2：`Blip2Processor` + `Blip2ForConditionalGeneration`（模型大，2.7B+）
- BLIP（轻量）：`BlipProcessor` + `BlipForConditionalGeneration`（~250MB）
- VQA：`BlipForQuestionAnswering`（视觉问答）

> 本上机用 BLIP-base 作为轻量替代。BLIP-2 API 几乎相同，仅类名和模型ID不同。

In [ ]:
# TODO 4：BLIP图文理解（transformers，产品图自动描述+VQA）
# 提示：用 BLIP 模型（BLIP-2的轻量版）实现：
#   1. 自动生成产品图片描述（image captioning）
#   2. 视觉问答（VQA）：对产品图片提问
# 模型：Salesforce/blip-image-captioning-base（~250MB，轻量）
#      BLIP-2对应：Salesforce/blip2-opt-2.7b（更大，API相似）
# 营销场景：自动为产品图片生成营销文案

# ===== 你的代码 =====
from transformers import BlipProcessor, BlipForConditionalGeneration

# TODO: 加载BLIP模型
# processor = ...
# model = ...

# TODO: 生成产品图片并自动描述
# TODO: 用VQA对产品图片提问

## 6. 跨域对齐与零样本分类

CLIP 的对齐空间天然支持零样本分类：无需训练，只需将类别名转为文本，用CLIP计算图片与各类别的相似度。

**零样本分类流程**：
1. 定义类别：["护肤品", "彩妆", "食品", ...]
2. 转为prompt：["a photo of skincare", ...]
3. CLIP编码图片+文本
4. 计算相似度 -> softmax -> 取最大值

**营销应用**：新品上架自动分类、广告图片自动打标签。

**跨域对齐的本质**：CLIP将图像和文本映射到同一空间，使"图片向量与文本向量的余弦相似度"直接反映语义匹配度。

In [ ]:
# TODO 5：跨域对齐与零样本分类（CLIP zero-shot）
# 提示：用CLIP实现零样本图片分类（无需训练即可分类）
#   1. 定义产品类别（护肤品/彩妆/食品/电子/服装）
#   2. 将类别转为文本prompt
#   3. 用CLIP计算图片与各类别的相似度
#   4. 选相似度最高的类别作为预测
# 营销场景：新品图片自动分类打标签

# ===== 你的代码 =====

# TODO: 定义产品类别
# categories = [...]

# TODO: 生成测试图片
# test_images = [...]

# TODO: 用CLIP进行零样本分类
# TODO: 打印分类结果和置信度
# TODO: 分析跨域对齐效果（正确匹配vs错误匹配的相似度差距）

## 7. 企业级多模态架构设计

将融合策略、对比学习、CLIP/BLIP整合为企业级系统。

**架构设计原则**（独立教材 3.4.3 节）：
1. **分层解耦**：编码层、融合层、对齐层、存储层各自独立
2. **多存储共存**：向量数据库（语义检索）+ 图数据库（关系推理）
3. **端到端可训练**：从编码到对齐可通过对比学习端到端优化
4. **在线/离线分离**：产品embedding预计算（离线），用户embedding实时计算（在线）

**本任务目标**：设计一个广告创意图文匹配系统，画出架构图，评估各模块。

In [ ]:
# TODO 6：企业级多模态架构设计（广告创意图文匹配系统）
# 提示：设计一个企业级广告创意图文匹配系统的架构
#   1. 画出架构图（ASCII art）
#   2. 定义各模块的模型/参数/延迟
#   3. 评估系统瓶颈和优化方向
# 营销场景：广告投放前，自动匹配广告文案与创意图片

# ===== 你的代码 =====

# TODO: 定义架构图（ASCII art）
# architecture = 

## 8. 反思与前沿

### 反思问题
1. 早融合 vs 晚融合：在什么场景下你会选哪个？（提示：模态是否总是同时可用？）
2. 温度参数 tau=0.01 vs tau=1.0，哪个更适合营销图文匹配？为什么？
3. CLIP零样本分类在什么情况下会失效？（提示：细粒度分类、领域偏移）
4. 从CLIP到GPT-4o，"原生多模态"解决了双塔架构的什么问题？

### 2026 前沿：原生多模态与对比学习
- **GPT-4o/Gemini**：端到端多模态训练，统一token空间处理文本/图像/音频
- **LLaVA**：开源视觉-语言模型（CLIP-ViT + 投影层 + LLM），低成本方案
- **对比学习仍是基础**：即使是原生多模态，预训练阶段仍大量使用对比学习
- **BLIP-3 / LLaVA-1.6**：2024-2025开源多模态持续进化

> 深入阅读见 reading.md 的 CLIP/BLIP-2/LLaVA 条目。